In [0]:
# ---------------------------------------------------------------------------
# GOLD LAYER, PART 1 — Harmonize BRFSS into common schema
# ---------------------------------------------------------------------------
from pyspark.sql import functions as F

silver_brfss = spark.table("workspace.default.silver_brfss_2024_clean")

# NOTE: standard BRFSS _RACE convention -- verify against this year's exact
# codebook value labels before publishing, not independently confirmed here
# the way NYTS's Q4A-G labels were.
brfss_race_map = {1: "White", 2: "Black", 3: "AIAN", 4: "Asian",
                   5: "NHPI", 6: "Other", 7: "Multiracial", 8: "Hispanic"}
race_map_expr = F.create_map([F.lit(x) for pair in brfss_race_map.items() for x in pair])

gold_brfss = silver_brfss.select(
    F.col("SEQNO").alias("person_id"),
    F.lit("BRFSS").alias("source_system"),
    F.lit(2024).alias("source_year"),
    F.lit("adult").alias("population"),
    F.col("_LLCPWT").alias("survey_weight"),
    F.col("SEXVAR").cast("int").alias("sex"),  # 1=Male, 2=Female -- same convention as NYTS, no mapping needed
    F.col("_AGEG5YR").alias("age_category_raw"),
    F.lit("BRFSS_5yr_bucket").alias("age_scale"),  # documents that this ISN'T the same scale as NYTS
    race_map_expr[F.col("_RACE")].alias("race_ethnicity"),
    F.when(F.col("_TOTINDA") == 2, 1).when(F.col("_TOTINDA") == 1, 0).alias("primary_outcome_flag"),
    F.lit("no_leisure_time_physical_activity").alias("primary_outcome_label"),
    F.col("_STATE").alias("geography"),
)

gold_brfss.write.format("delta").mode("overwrite").saveAsTable("workspace.default.gold_brfss_2024_harmonized")
print(f"gold_brfss_2024_harmonized: {gold_brfss.count()} rows")
gold_brfss.show(5)

gold_brfss_2024_harmonized: 416081 rows
+----------+-------------+-----------+----------+-------------+---+----------------+----------------+--------------+--------------------+---------------------+---------+
| person_id|source_system|source_year|population|survey_weight|sex|age_category_raw|       age_scale|race_ethnicity|primary_outcome_flag|primary_outcome_label|geography|
+----------+-------------+-----------+----------+-------------+---+----------------+----------------+--------------+--------------------+---------------------+---------+
|2024000001|        BRFSS|       2024|     adult|   261.525511|  2|              12|BRFSS_5yr_bucket|         White|                   0| no_leisure_time_p...|       01|
|2024000002|        BRFSS|       2024|     adult|   307.169688|  1|              13|BRFSS_5yr_bucket|         White|                   0| no_leisure_time_p...|       01|
|2024000003|        BRFSS|       2024|     adult|   2939.86281|  1|               8|BRFSS_5yr_bucket|         

In [0]:
# ---------------------------------------------------------------------------
# GOLD LAYER, PART 2 — Harmonize NYTS into the same common schema
# Race/ethnicity requires real transformation here: BRFSS gives one
# categorical field, NYTS gives 7 independent select-all-that-apply flags.
# Collapsing to a single category (with "Multiracial" when >1 flag selected)
# is the actual harmonization work -- not a formality.
# ---------------------------------------------------------------------------
silver_nyts = spark.table("workspace.default.silver_nyts_2025_clean")

race_cols = ["Q4A", "Q4B", "Q4C", "Q4D", "Q4E", "Q4F", "Q4G"]
race_labels = {"Q4A": "AIAN", "Q4B": "Asian", "Q4C": "Black", "Q4D": "Hispanic",
                "Q4E": "MENA", "Q4F": "NHPI", "Q4G": "White"}

race_count_expr = sum(F.coalesce(F.col(c), F.lit(0.0)) for c in race_cols)

race_case = F.when(race_count_expr > 1, "Multiracial").when(race_count_expr == 0, "Unknown")
for c in race_cols:
    race_case = race_case.when((F.col(c) == 1.0) & (race_count_expr == 1), race_labels[c])
race_case = race_case.otherwise("Unknown")

gold_nyts = silver_nyts.select(
    F.col("ARTIFICIAL_ID").alias("person_id"),
    F.lit("NYTS").alias("source_system"),
    F.lit(2025).alias("source_year"),
    F.lit("youth").alias("population"),
    F.col("WT_ANALYSIS").cast("double").alias("survey_weight"),
    F.col("Q2").cast("int").alias("sex"),  # 1=Male, 2=Female -- matches BRFSS convention directly
    F.col("Q1").alias("age_category_raw"),
    F.lit("NYTS_single_year").alias("age_scale"),  # different scale than BRFSS -- documented, not silently merged
    race_case.alias("race_ethnicity"),
    F.when(F.col("CELCIGT") == 1.0, 1).when(F.col("CELCIGT") == 2.0, 0).alias("primary_outcome_flag"),
    F.lit("current_ecigarette_use").alias("primary_outcome_label"),
    F.lit(None).cast("string").alias("geography"),  # NYTS public files don't provide state-level identifiers
)

gold_nyts.write.format("delta").mode("overwrite").saveAsTable("workspace.default.gold_nyts_2025_harmonized")
print(f"gold_nyts_2025_harmonized: {gold_nyts.count()} rows")
gold_nyts.show(5)

gold_nyts_2025_harmonized: 23380 rows
+---------+-------------+-----------+----------+------------------+---+----------------+----------------+--------------+--------------------+---------------------+---------+
|person_id|source_system|source_year|population|     survey_weight|sex|age_category_raw|       age_scale|race_ethnicity|primary_outcome_flag|primary_outcome_label|geography|
+---------+-------------+-----------+----------+------------------+---+----------------+----------------+--------------+--------------------+---------------------+---------+
|A25000007|         NYTS|       2025|     youth| 1938.587323667018|  2|             4.0|NYTS_single_year|         White|                   0| current_ecigarett...|     NULL|
|A25000018|         NYTS|       2025|     youth|2051.3293367094525|  1|            11.0|NYTS_single_year|       Unknown|                   0| current_ecigarett...|     NULL|
|A25000021|         NYTS|       2025|     youth| 1767.210026210859|  1|             4.0|NYTS

In [0]:
gold_brfss = spark.table("workspace.default.gold_brfss_2024_harmonized")
gold_nyts = spark.table("workspace.default.gold_nyts_2025_harmonized")

print("BRFSS race_ethnicity distribution:")
gold_brfss.groupBy("race_ethnicity").count().orderBy(F.desc("count")).show()

print("NYTS race_ethnicity distribution:")
gold_nyts.groupBy("race_ethnicity").count().orderBy(F.desc("count")).show()

BRFSS race_ethnicity distribution:
+--------------+------+
|race_ethnicity| count|
+--------------+------+
|         White|308670|
|      Hispanic| 43754|
|         Black| 31735|
|         Asian| 11437|
|   Multiracial|  9608|
|          AIAN|  5775|
|         Other|  3217|
|          NHPI|  1885|
+--------------+------+

NYTS race_ethnicity distribution:
+--------------+-----+
|race_ethnicity|count|
+--------------+-----+
|         White| 9645|
|      Hispanic| 4639|
|   Multiracial| 3146|
|         Black| 2895|
|          AIAN| 1423|
|         Asian| 1102|
|          MENA|  298|
|       Unknown|  166|
|          NHPI|   66|
+--------------+-----+



In [0]:
# ---------------------------------------------------------------------------
# GOLD LAYER, PART 3 — Reusable cross-system prevalence function
# Works identically against gold_brfss_2024_harmonized and
# gold_nyts_2025_harmonized because both share the same schema.
# This function is the actual interoperability deliverable: one piece of
# logic, zero source-specific branches, usable on any future CDC dataset
# harmonized into this same schema.
# ---------------------------------------------------------------------------
def weighted_prevalence_by_group(table_name, group_col):
    """
    Computes weighted prevalence of primary_outcome_flag, broken down by
    a demographic column, for any table conforming to the harmonized schema.
    """
    df = spark.table(table_name)
    result = df.groupBy(group_col).agg(
        (F.sum(F.when(F.col("primary_outcome_flag") == 1, F.col("survey_weight")).otherwise(0)) /
         F.sum("survey_weight") * 100).alias("weighted_prevalence_pct"),
        F.count("*").alias("n"),
        F.first("source_system").alias("source_system"),
        F.first("primary_outcome_label").alias("outcome"),
    )
    return result.orderBy(F.desc("weighted_prevalence_pct"))

print("=== BRFSS: physical inactivity prevalence by race/ethnicity ===")
weighted_prevalence_by_group("workspace.default.gold_brfss_2024_harmonized", "race_ethnicity").show()

print("=== NYTS: e-cigarette use prevalence by race/ethnicity ===")
weighted_prevalence_by_group("workspace.default.gold_nyts_2025_harmonized", "race_ethnicity").show()

print("=== BRFSS: physical inactivity prevalence by sex ===")
weighted_prevalence_by_group("workspace.default.gold_brfss_2024_harmonized", "sex").show()

print("=== NYTS: e-cigarette use prevalence by sex ===")
weighted_prevalence_by_group("workspace.default.gold_nyts_2025_harmonized", "sex").show()

=== BRFSS: physical inactivity prevalence by race/ethnicity ===
+--------------+-----------------------+------+-------------+--------------------+
|race_ethnicity|weighted_prevalence_pct|     n|source_system|             outcome|
+--------------+-----------------------+------+-------------+--------------------+
|      Hispanic|      29.00004629375034| 43754|        BRFSS|no_leisure_time_p...|
|         Black|     25.478308625484576| 31735|        BRFSS|no_leisure_time_p...|
|          AIAN|       25.1199870542873|  5775|        BRFSS|no_leisure_time_p...|
|         Other|      22.80193340089075|  3217|        BRFSS|no_leisure_time_p...|
|          NHPI|      20.09503543583398|  1885|        BRFSS|no_leisure_time_p...|
|         White|      19.47079856114838|308670|        BRFSS|no_leisure_time_p...|
|   Multiracial|     17.744112378603806|  9608|        BRFSS|no_leisure_time_p...|
|         Asian|      17.18998437631942| 11437|        BRFSS|no_leisure_time_p...|
+--------------+-------